In [23]:
import json

from openai import OpenAI
from dotenv import load_dotenv
import requests
import os
import time

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

openai = OpenAI(api_key=OPENAI_API_KEY)

In [24]:
def get_website_html(url: str) -> str:
    try:
        response = requests.get(url)
        response.raise_for_status()
        return response.text
    except requests.exceptions.RequestException as e:
        print(f"Error fetching website content: {e}")
        return ""

In [25]:
def extract_core_website_content(html: str) -> str:
    prompt = f"""
    You are an expert web content extractor. Your task is to extract the core content from a given HTML page.
    The core content should be the main text, excluding navigation, footers, and other non-esential elements like scripts, styles, etc.

    Here is the html content of the page:
    <HTML>
    {html}
    </HTML>
    Extract the content and return it as plain text.
    """

    model = "gpt-4o"
    response = openai.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content.strip()

In [26]:
def summarise_content(content: str) -> str:
    prompt = f"""
    You are an expert summarizer. Your task is to read the following content and provide a concise summary that captures the main points and key information.

    Here is the content to summarize:
    <CONTENT>
    {content}
    </CONTENT>

    Provide a clear and concise summary of the above content. Prefer bullet points and av oid unnecessary explanations. 
    Focus on the most important information and insights.
    """

    model = "gpt-4o"
    response = openai.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content.strip()

In [27]:
def extract_topics(content: str, max_tokens: int = 100) -> str:
    prompt = f"""
    You are an expert topic extractor. Your task is to read the following content and extract the main topic discussed in it.

    Here is the content to analyze:
    <CONTENT>
    {content}
    </CONTENT>

    Extract the main topic from the above content and return it as a concise bullet point. Focus on the most important theme or subject covered in the content.
    """

    model = "gpt-4o"
    response = openai.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt}
        ],
        max_tokens=max_tokens
    )
    topic = response.choices[0].message.content.strip()
    return topic

In [28]:
def generate_x_post(prompt:str, system_prompt: str) -> str:
    model = "gpt-4o-mini"
    max_tokens = 100
    response = openai.chat.completions.create(
        model=model,
        messages=[
            {"role": "user", "content": prompt},
            {"role": "system", "content": system_prompt}
        ],
        max_tokens=max_tokens
    )
    return response.choices[0].message.content.strip()

In [29]:
def get_prompt(topic: str) -> str:
    with open("post_examples.json", "r") as f:
        examples = json.load(f)
    examples_str = ""

    for i, example in enumerate(examples):
        examples_str += f"""
        <example-{i+1}>
            <topic>
                {example['topic']}
            </topic>
            <generate-post>
                {example['post']}
            </generate-post>
        </example-{i+1}>
        """

    prompt=f"""
        You are an expert social media manager, and you excel at crafting viral and highly engaging posts for X (formerly Twitter).

        Your task is to generate a post that is concise, impactful, and tailored to the topic provided by the user.

        Avoid using hashtags and lots of emojis (a few emojis are okay, but not too many).

        Keep the post short and focused, structure it in a clean, readable way, using line breaks and empty lines to enhance readability.
        The topic for the post is:
        <topic>
            {topic}
        </topic>

        <examples>
            {examples_str}
        </examples>

        Use the tone, language, structure and style of the examples above to generate a post that is engaging and relevant to the topic provided by the user.
        DO NOT USE THE CONTENT FROM THE EXAMPLES!
        """

    return prompt

In [30]:
#extract responses to file
def save_response_to_file(response: str, filename: str) -> None:
    with open(filename, "w") as f:
        f.write(response)

In [31]:
website_url = input("Enter the website URL: ")
print("Fetching website content...")

system_prompt = """
    You are an expert social media manager, and you excel at crafting viral and highly engaging posts for X (formerly Twitter). 
    Generate a viral and engaging post for X(formerly Twitter) based on the provided topic. 
    Avoid using hashtags and lots of emojis (a few emojis are okay, but not too many). 
    Keep the post short and focused, structure it in a clean, readable way, using line breaks and empty lines to enhance readability.
    """

try:
    session_id = int(time.time())
    html_content = get_website_html(website_url)
    save_response_to_file(html_content, f"responses/00_html_content_{session_id}.txt")
    print("Extracting core content from the website...")
    core_content = extract_core_website_content(html_content)
    save_response_to_file(core_content, f"responses/01_core_content_{session_id}.txt")
    print("Core content extracted successfully.")
    print("-" * 50)
    print("Summarizing the extracted content...")
    summary = summarise_content(core_content)
    save_response_to_file(summary, f"responses/02_summary_{session_id}.txt")
    print("Summary generated successfully.")
    print("-" * 50)
    print("Generating X post based on the summary...")
    prompt = get_prompt(summary)
    topic = extract_topics(summary)
    x_post = generate_x_post(prompt, system_prompt)
    topic_slug = topic.replace(" ", "_").lower().replace("/", "_")
    save_response_to_file(x_post, f"responses/03_generated_post_{topic_slug}_{session_id}.txt")
    print("\nGenerated X Post:")
    print(x_post)

except Exception as e:
    print(f"An error occurred: {e}")

Fetching website content...
Extracting core content from the website...
Core content extracted successfully.
--------------------------------------------------
Summarizing the extracted content...
Summary generated successfully.
--------------------------------------------------
Generating X post based on the summary...

Generated X Post:
Meet Paul: an expert at turning ideas into AI-driven realities.

With a track record in full-stack development and software architecture, he crafts scalable microservices and enterprise applications that deliver real impact. 

His projects include:

- A React.js Playground mastering POCs and integrations.

- A sleek Personal Portfolio showcasing skills and creativity.

What sets Paul apart? 

Expertise in AI workflows, cloud platforms like AWS, and CI/CD strategies. 

He doesn’t just code; he leads, communicates, and constantly
